# Experiment 6: CGM + time_since_meal + image (dinov2, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [3]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+dinov2 (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'dinov2'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_dinov2'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_dinov2_stability_summary.json'

In [4]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [5]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_dinov2' / f'gated_cgm_time_dinov2_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_dinov2_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM+time+dinov2 (gated) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.69 | MAE 9.20 | gRMSE 15.85 | Zone A 95.01% | A+B 99.59%
 30 min | RMSE 20.08 | MAE 13.21 | gRMSE 24.04 | Zone A 89.49% | A+B 99.41%
 60 min | RMSE 28.81 | MAE 18.70 | gRMSE 35.87 | Zone A 80.68% | A+B 98.70%
 90 min | RMSE 34.20 | MAE 22.37 | gRMSE 43.20 | Zone A 73.98% | A+B 97.91%
120 min | RMSE 37.51 | MAE 24.73 | gRMSE 47.90 | Zone A 70.38% | A+B 97.25%

--------------------------------------------------------------------------------
Test results — CGM+time+dinov2 (gated) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.68 | MAE 8.43 | gRMSE 14.67 | Zone A 96.19% | A+B 99.76%
 30 min | RMSE 19.85 | MAE 12.95 | gRMSE 23.93 | Zone A 89.63% | A+B 99.44%
 60 min | RMSE 29.18 | MAE 18.77 | gRMSE 36.53 | Zone A 80.25% | A+B 98.54%
 90 min | RMSE 34.94 | MAE 22.63 | gRMSE 44.42 | Zone A 74.09% | A+B 97.80%
120 min | RMSE 38.26 | MAE 25.09 | gRMSE 49.00 | Zone A 69.79% | A+B 97.18%

--------------------------------------------------------------------------------
Test results — CGM+time+dinov2 (gated) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 12.19 | MAE 8.24 | gRMSE 13.76 | Zone A 96.06% | A+B 99.74%
 30 min | RMSE 19.17 | MAE 12.62 | gRMSE 22.67 | Zone A 89.92% | A+B 99.50%
 60 min | RMSE 28.29 | MAE 18.47 | gRMSE 34.97 | Zone A 80.22% | A+B 98.84%
 90 min | RMSE 34.11 | MAE 22.38 | gRMSE 43.05 | Zone A 73.92% | A+B 98.12%
120 min | RMSE 37.58 | MAE 24.70 | gRMSE 47.95 | Zone A 70.63% | A+B 97.37%

--------------------------------------------------------------------------------
Test results — CGM+time+dinov2 (gated) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.49 | MAE 8.96 | gRMSE 15.14 | Zone A 94.82% | A+B 99.65%
 30 min | RMSE 19.88 | MAE 12.99 | gRMSE 23.39 | Zone A 89.13% | A+B 99.44%
 60 min | RMSE 28.66 | MAE 18.62 | gRMSE 35.42 | Zone A 80.42% | A+B 98.79%
 90 min | RMSE 34.55 | MAE 22.41 | gRMSE 43.78 | Zone A 74.30% | A+B 97.81%
120 min | RMSE 37.77 | MAE 24.75 | gRMSE 48.32 | Zone A 70.52% | A+B 97.14%

--------------------------------------------------------------------------------
Test results — CGM+time+dinov2 (gated) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 13.60 | MAE 9.04 | gRMSE 15.57 | Zone A 95.04% | A+B 99.65%
 30 min | RMSE 20.19 | MAE 13.17 | gRMSE 24.10 | Zone A 89.44% | A+B 99.41%
 60 min | RMSE 28.92 | MAE 18.69 | gRMSE 35.93 | Zone A 80.76% | A+B 98.69%
 90 min | RMSE 34.43 | MAE 22.33 | gRMSE 43.54 | Zone A 74.60% | A+B 97.94%
120 min | RMSE 37.57 | MAE 24.67 | gRMSE 47.87 | Zone A 70.86% | A+B 97.28%

--------------------------------------------------------------------------------


In [7]:
# Test Results:
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM+time+dinov2 (gated) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 13.25 +/- 0.89 (2.62) mg/dL   MAE = 8.85 +/- 0.59 (1.99)   gRMSE = 15.20 +/- 1.21 (4.43)   Zone A = 95.55 +/- 0.85 (2.97)%   Zone A+B = 99.70 +/- 0.10 (0.49)%
 30-min:  RMSE = 19.97 +/- 0.59 (4.98) mg/dL   MAE = 13.08 +/- 0.39 (3.45)   gRMSE = 23.84 +/- 0.91 (8.39)   Zone A = 89.50 +/- 0.43 (3.93)%   Zone A+B = 99.43 +/- 0.06 (0.72)%
 60-min:  RMSE = 28.94 +/- 0.43 (10.35) mg/dL   MAE = 18.70 +/- 0.18 (6.42)   gRMSE = 36.02 +/- 0.73 (16.55)   Zone A = 80.49 +/- 0.45 (5.17)%   Zone A+B = 98.68 +/- 0.13 (1.41)%
 90-min:  RMSE = 34.54 +/- 0.27 (14.23) mg/dL   MAE = 22.42 +/- 0.13 (8.46)   gRMSE = 43.75 +/- 0.48 (22.40)   Zone A = 74.33 +/- 0.44 (7.14)%   Zone A+B = 97.91 +/- 0.12 (2.56)%
120-min:  RMSE = 37.72 +/- 0.22 (16.64) mg/dL   MAE = 24.75 +/- 0.14 (9.86)   gRMSE = 48.16 +/- 0.37 (26.05)   Zone A = 70.61 +/- 0.38 (8.22)%   Zone A+B = 97.32 +/- 0.22 (3.84)%

saved per-seed results to